In [ ]:
!pip install transformers sentence_transformers datasets sacrebleu

In [ ]:
from transformers import MarianMTModel, MarianTokenizer,AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer, util
from datasets import load_dataset, concatenate_datasets
import pandas as pd
import torch,os
from tqdm import tqdm
import sacrebleu
from sacrebleu.metrics import BLEU, CHRF

In [ ]:
_bleu_metric = BLEU(effective_order=True)
_chrf_metric = CHRF()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
file_path_a1a2 = '/content/drive/MyDrive/Data/A1_mynediad-A2_sylfaen-de-learnwelsh.json'

In [ ]:
df_a1_a2 = pd.read_json(file_path_a1a2)

In [ ]:
df_a1_a2.head()

In [ ]:
df_a1_a2["source_name"].value_counts()

In [ ]:
mynediad_df = df_a1_a2[df_a1_a2['source_name'] == 'mynediad-de-learnwelsh']
sylfaen_df = df_a1_a2[df_a1_a2['source_name'] == 'sylfaen-de-learnwelsh']

# Get CEFR levels present
mynediad_level = mynediad_df['cefr_level'].iloc[0]
sylfaen_level = sylfaen_df['cefr_level'].iloc[0]

# Create JSON filenames dynamically
mynediad_filename = f"{mynediad_level}_mynediad-de-learnwelsh.json"
sylfaen_filename = f"{sylfaen_level}_sylfaen-de-learnwelsh.json"

# Export both as JSON (UTF-8, human-readable)
mynediad_df.to_json(mynediad_filename, orient='records', force_ascii=False, indent=2)
sylfaen_df.to_json(sylfaen_filename, orient='records', force_ascii=False, indent=2)

In [ ]:
file_paths = [
    '/content/drive/MyDrive/Data/B1_canolradd_de-learnwelsh.json',
    '/content/drive/MyDrive/Data/B2_uwch-1_de-learnwelsh.json',
    '/content/drive/MyDrive/Data/B2_Uwch-2_de-learnwelsh.json',
    '/content/drive/MyDrive/Data/B2_Uwch-3_de-learnwelsh.json'
]

In [ ]:
# Read and combine all B1-B2 data
df_b1_b2_list = []

for path in file_paths:
    df = pd.read_json(path)
    df = df[df["cefr_level"].isin(["B1", "B2"])]  # filter by CEFR level
    df = df[["text", "cefr_level"]].dropna().drop_duplicates(subset="text")
    df_b1_b2_list.append(df)

# Combine all DataFrames into one
df_b1_b2 = pd.concat(df_b1_b2_list, ignore_index=True)

# Reset index for cleanliness
df_b1_b2 = df_b1_b2.reset_index(drop=True)

print(f"Combined B1-B2 dataset shape: {df_b1_b2.shape}")
df_b1_b2.head()

In [ ]:
df_b1_b2['cefr_level'].value_counts()

In [ ]:
# Batched pipeline
records = []
batch_size = 64   # tune based on GPU RAM; try 64/128 on A100, fall back if OOM

In [ ]:
texts = df_b1_b2["text"].tolist()
levels = df_b1_b2["cefr_level"].tolist()

In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cuda.matmul.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

# Prefer bfloat16 on A100; fall back to float16 elsewhere
AMP_DTYPE = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8) else torch.float16

cy2en_name = "Helsinki-NLP/opus-mt-cy-en"
en2cy_name = "Helsinki-NLP/opus-mt-en-cy"

# Fast tokenizers + explicit settings
cy2en_tokenizer = AutoTokenizer.from_pretrained(cy2en_name, use_fast=True)
en2cy_tokenizer = AutoTokenizer.from_pretrained(en2cy_name, use_fast=True)

# Load with mixed precision, SDPA attention if supported, and low CPU mem
common_model_kwargs = dict(
    torch_dtype=AMP_DTYPE,
    low_cpu_mem_usage=True,
)

# Try SDPA / Flash-style attention (safe to ignore if unsupported)
try:
    common_model_kwargs["attn_implementation"] = "sdpa"
except Exception:
    pass

cy2en_model = AutoModelForSeq2SeqLM.from_pretrained(cy2en_name, **common_model_kwargs).to(device)
en2cy_model = AutoModelForSeq2SeqLM.from_pretrained(en2cy_name, **common_model_kwargs).to(device)

# (Optional) BetterTransformer fuse for faster encoder/decoder attention
# Works on many encoder-decoder models; skip silently if not available.
try:
    cy2en_model = cy2en_model.to_bettertransformer()
    en2cy_model  = en2cy_model.to_bettertransformer()
except Exception:
    pass

# (Optional) torch.compile can help a bit after a warmup
try:
    cy2en_model = torch.compile(cy2en_model)
    en2cy_model = torch.compile(en2cy_model)
except Exception:
    pass

# Sensible generation defaults (override per-call as needed)
cy2en_model.generation_config.do_sample = False
cy2en_model.generation_config.num_beams = 1
cy2en_model.generation_config.use_cache = True
cy2en_model.generation_config.max_new_tokens = 128

en2cy_model.generation_config = cy2en_model.generation_config

cy2en_model.eval()
en2cy_model.eval()

In [ ]:
@torch.inference_mode()
def translate_batch(texts, tokenizer, model, max_new_tokens=128):
    enc = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
        pad_to_multiple_of=8,   # small perf win on A100
    )
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.batch_decode(out, skip_special_tokens=True)


In [ ]:
@torch.inference_mode()
def semantic_similarity_batch(ref_texts, hyp_texts, model=None, batch_size=256, normalize=True):
    """
    Pairwise cosine similarity between aligned lists of texts.

    Args:
        ref_texts (list[str]): reference/original sentences
        hyp_texts (list[str]): hypothesis/back-translated sentences
        model (SentenceTransformer | None): if None, uses global `sim_model`
        batch_size (int): encoding batch size for the SentenceTransformer
        normalize (bool): if True, uses normalized embeddings and fast dot product

    Returns:
        list[float]: cosine similarities for each (ref, hyp) pair
    """
    if model is None:
        global sim_model
        model = sim_model

    if len(ref_texts) != len(hyp_texts):
        raise ValueError(f"Lengths differ: {len(ref_texts)} refs vs {len(hyp_texts)} hyps")
    if len(ref_texts) == 0:
        return []

    # Safe casting in case there are None/NaN values
    refs = ["" if r is None else str(r) for r in ref_texts]
    hyps = ["" if h is None else str(h) for h in hyp_texts]

    emb_ref = model.encode(
        refs, convert_to_tensor=True, batch_size=batch_size,
        normalize_embeddings=normalize
    )
    emb_hyp = model.encode(
        hyps, convert_to_tensor=True, batch_size=batch_size,
        normalize_embeddings=normalize
    )

    if normalize:
        # For unit-normalized embeddings, cosine = dot product
        cos = (emb_ref * emb_hyp).sum(dim=1)
    else:
        # General cosine; take the diagonal of the full sim matrix
        cos = util.cos_sim(emb_ref, emb_hyp).diagonal()

    return cos.float().cpu().numpy().tolist()


In [ ]:
def evaluate_batch(
    ref_texts,
    hyp_texts,
    round_digits=(2, 2, 4),
    sim_batch_size=256,
    normalize_cosine=True,
):
    """
    Compute per-sample BLEU, chrF, and cosine similarity.
    Returns three lists: bleu_list, chrf_list, cosine_list

    Args:
        ref_texts: list[str] reference/original sentences
        hyp_texts: list[str] hypothesis/back-translated sentences
        round_digits: (bleu, chrf, cosine) decimals or None to skip rounding
        sim_batch_size: batch size for sentence-transformer encoding on GPU
        normalize_cosine: if True, uses normalized embeddings for fast dot-product cosine
    """
    # --- validations & edge cases ---
    if len(ref_texts) != len(hyp_texts):
        raise ValueError(f"Lengths differ: {len(ref_texts)} refs vs {len(hyp_texts)} hyps")
    if len(ref_texts) == 0:
        return [], [], []

    # Cast to safe strings in case of None/NaN
    refs = ["" if r is None else str(r) for r in ref_texts]
    hyps = ["" if h is None else str(h) for h in hyp_texts]

    # --- BLEU / chrF sentence-level (sacrebleu scorers created once globally) ---
    bleu_list = [_bleu_metric.sentence_score(h, [r]).score for r, h in zip(refs, hyps)]
    chrf_list = [_chrf_metric.sentence_score(h, [r]).score for r, h in zip(refs, hyps)]

    # --- Cosine similarity (batched on GPU) ---
    cosine_list = semantic_similarity_batch(refs, hyps, batch_size=sim_batch_size, normalize=normalize_cosine)

    # --- rounding (optional) ---
    if round_digits is not None:
        b, c, s = round_digits
        bleu_list   = [round(x, b) for x in bleu_list]
        chrf_list   = [round(x, c) for x in chrf_list]
        cosine_list = [round(float(x), s) for x in cosine_list]

    return bleu_list, chrf_list, cosine_list


In [ ]:
records = []

i = 0
pbar = tqdm(total=len(texts), desc="Back-translating (batched)")

while i < len(texts):
    end = min(i + batch_size, len(texts))
    batch_texts  = texts[i:end]
    batch_levels = levels[i:end]

    try:
        # cy -> en (batch)
        batch_en = translate_batch(batch_texts, cy2en_tokenizer, cy2en_model)

        # en -> cy (batch)
        batch_back_cy = translate_batch(batch_en, en2cy_tokenizer, en2cy_model)

        # ---- Batched evaluation (BLEU, chrF, cosine) ----
        bleu_list, chrf_list, cosine_list = evaluate_batch(batch_texts, batch_back_cy)

        # Collect results
        for original_cy, english, back_cy, cefr, bleu, chrf, cosine in zip(
            batch_texts, batch_en, batch_back_cy, batch_levels, bleu_list, chrf_list, cosine_list
        ):
            records.append({
                "original_welsh": original_cy,
                "translated_english": english,
                "back_translated_welsh": back_cy,
                "cefr_level": cefr,
                "BLEU": bleu,
                "chrF": chrf,
                "cosine_similarity": cosine
            })

        pbar.update(len(batch_texts))
        i = end  # advance only after a successful batch

    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        new_bs = max(8, batch_size // 2)
        if new_bs == batch_size:
            print("OOM but cannot reduce batch size further; skipping this batch.")
            i = end  # skip to avoid infinite loop
        else:
            batch_size = new_bs
            print(f"OOM — reducing batch_size to {batch_size} and retrying this batch.")
        # don't advance i here if retrying

    except Exception as e:
        print(f"Batch error on items {i}:{end}: {e}")
        i = end  # skip problematic batch and continue

pbar.close()

In [ ]:
df_bt = pd.DataFrame.from_records(records)
filtered_df = df_bt[df_bt["cosine_similarity"] > 0.80]
output_path = '/content/drive/MyDrive/Data/welsh_back_translation_B1_B2__high_similarity.csv'
filtered_df.to_csv(output_path, index=False)